# Ask Spurgeon - Full Fine-Tuning Pipeline

**Goal**: Fine-tune Llama-3.1-8B with **high fidelity** to Spurgeon's actual sermons using QLoRA + Unsloth.

This notebook implements the complete pipeline:
1. Generate high-quality, grounded synthetic data (using your existing RAG + Groq)
2. Format into Llama-3 ChatML
3. Train with Unsloth QLoRA (optimized for T4 / A100)
4. Merge and save the adapter

**Recommended Runtime**: Google Colab Pro (A100) or T4 (free tier - smaller experiments only)

## 1. Setup & Installation

In [ ]:
# Install Unsloth (must be done first)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl" peft accelerate bitsandbytes

# Other dependencies
!pip install groq chromadb llama-index-vector-stores-chroma llama-index-embeddings-huggingface python-dotenv tqdm

In [ ]:
# Clone the repository (contains your RAG code and fine-tuning scripts)
!git clone https://github.com/elViRafa/ask-spurgeon.git
%cd ask-spurgeon

## 2. Configuration

In [ ]:
import os
from pathlib import Path

# === REQUIRED: Your Groq API Key for synthetic data generation ===
os.environ["GROQ_API_KEY"] = "gsk_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  # <-- REPLACE THIS

# === Paths ===
OUTPUT_DIR = "./fine_tuning/outputs/spurgeon-8b-qlora"
DATASET_PATH = "./fine_tuning/data/spurgeon_train_1500.jsonl"  # Using the prepared 1500-example high-fidelity dataset

Path("fine_tuning/data").mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")

## 3. Generate Synthetic Training Data (High Fidelity)

This step uses your existing RAG system + Groq 70B to create grounded Q&A pairs in Spurgeon's voice.

**Note**: This can take time and costs Groq credits. Start small (500-1000 examples).

In [ ]:
# Option A: Generate new data (recommended to start small)
!python fine_tuning/scripts/generate_synthetic_data.py \
    --limit 800 \
    --output {DATASET_PATH}

In [ ]:
# Option B: Upload your own dataset instead
# from google.colab import files
# uploaded = files.upload()  # Upload your .jsonl file
# DATASET_PATH = list(uploaded.keys())[0]

## 4. Train with Unsloth QLoRA

In [ ]:
!python fine_tuning/scripts/train_spurgeon_qlora.py \
    --dataset {DATASET_PATH} \
    --output-dir {OUTPUT_DIR} \
    --save-merged

## 5. (Optional) Merge & Save Final Model

In [ ]:
from unsloth import FastLanguageModel

print("Merging LoRA adapter into base model...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=OUTPUT_DIR,
    max_seq_length=4096,
    load_in_4bit=True,
)

model.save_pretrained_merged(
    f"{OUTPUT_DIR}-merged-16bit",
    tokenizer,
    save_method="merged_16bit",
)

print(f"Merged model saved to: {OUTPUT_DIR}-merged-16bit")

## 6. Test the Fine-Tuned Model (Inference)

Now that the model is trained, let's verify that it actually speaks in Spurgeon's voice while remaining **faithful** to the retrieved context.

We will load a test context and question, format it using Llama-3's chat template, and use Unsloth's fast inference with **real-time streaming** to generate the response.

In [ ]:
# @title 6. Real-Time Inference Test
from unsloth import FastLanguageModel
from transformers import TextStreamer

# 1. Set model to fast inference mode
FastLanguageModel.for_inference(model)

# 2. Prepare a test sermon context and a question
test_context = (
    "In my sermon 'Songs in the Night', I declared: 'Any man can sing in the day. "
    "It is easy to sing when we can read the notes by daylight; but the skillful singer "
    "is he who can sing when there is not a ray of light to read by. Songs in the night come "
    "only from God; they are not in the power of man. They are the sovereign gifts of heaven.'"
)
test_question = "Where do songs in the night come from, and is it easy for man to sing them?"

# 3. Format using Llama-3 chat template (matching the training format)
messages = [
    {
        "role": "system",
        "content": (
            "You are Charles Haddon Spurgeon. You must answer the user's question "
            "using ONLY the information in the CONTEXT section below. Do not add external knowledge.\n\n"
            f"CONTEXT:\n{test_context}"
        )
    },
    {
        "role": "user",
        "content": test_question
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

# 4. Setup text streamer for real-time output
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("=== Retrieved Context ===")
print(test_context)
print("\n=== User Question ===")
print(test_question)
print("\n=== Generated Answer (Spurgeon's Voice) ===")

# 5. Generate with streaming
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 256,
    use_cache = True,
    temperature = 0.6,
    top_p = 0.9,
)

## 7. Upload to Hugging Face (Recommended)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # Login with your HF token

In [ ]:
# Upload merged model
from huggingface_hub import upload_folder

repo_id = "YOUR_USERNAME/llama-3.1-8b-spurgeon-fidelity"  # Change this!

upload_folder(
    folder_path=f"{OUTPUT_DIR}-merged-16bit",
    repo_id=repo_id,
    commit_message="Upload fine-tuned Spurgeon generator (high fidelity)",
)

## Next Steps After Training

1. **Test the model** locally or on HF Inference Endpoints.
2. Integrate it into your RAG pipeline as the generator (instead of or in addition to Groq).
3. Run proper evaluation comparing faithfulness vs your current system.
4. Consider training a 70B version later if results are promising.

See `fine_tuning/FINE_TUNING_PLAN.md` for the full strategy and evaluation recommendations.